# 05. Model Evaluation
Welcome to the Model Evaluation phase of our Machine Learning pipeline!

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc
import logging

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 2. Load Models & Data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

def load_data_and_models():
    df = pd.read_csv("../Titanic-Dataset.csv")
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
    df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)
    features = ["Pclass", "Sex", "Age", "FamilySize", "Fare", "Embarked_Q", "Embarked_S"]
    X = df[features]
    y = df["Survived"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    log_reg = LogisticRegression(max_iter=1000, random_state=42)
    log_reg.fit(X_train, y_train)
    tree_clf = DecisionTreeClassifier(max_depth=4, random_state=42)
    tree_clf.fit(X_train, y_train)
    return X_test, y_test, log_reg, tree_clf

X_test, y_test, loaded_log_reg, loaded_tree_clf = load_data_and_models()

## 3. Evaluation Functions

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    metrics = {"Model": model_name, "Accuracy": accuracy_score(y_test, y_pred), "Precision": precision_score(y_test, y_pred), "Recall": recall_score(y_test, y_pred), "F1 Score": f1_score(y_test, y_pred)}
    for k, v in metrics.items():
        if k != "Model": print(f"{k}: {v:.4f}")
    return metrics

def plot_confusion_matrix(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{model_name} Confusion Matrix")
    plt.show()

## 4. Run Evaluation

In [ ]:
print("Logistic Regression")
log_metrics = evaluate_model(loaded_log_reg, X_test, y_test, "Logistic")
print("\nDecision Tree")
tree_metrics = evaluate_model(loaded_tree_clf, X_test, y_test, "Tree")
log_y_pred = loaded_log_reg.predict(X_test)
tree_y_pred = loaded_tree_clf.predict(X_test)

## 5. Visualizations

In [ ]:
plot_confusion_matrix(y_test, log_y_pred, "Logistic Regression")
plot_confusion_matrix(y_test, tree_y_pred, "Decision Tree")